In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset    # to get CIFAR-10 data from HuggingFace
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.kernel_pca import RFF_KernelPCA
from src.utils.metrics import metrics
import src.utils.plotting_params
from src.utils import misc 
from pathlib import Path


# automatic reload
%load_ext autoreload
%autoreload 2

In [ ]:
out_dir = "../../writeup/Kernel_PCA_Preimage/"

### Download CIFAR-10 data

In [ ]:
ds = load_dataset("cifar10")

X_train = np.array(ds["train"]["img"]).reshape(-1, 3072) / 255.0
y_train = np.array(ds["train"]["label"])

X_test  = np.array(ds["test"]["img"]).reshape(-1, 3072) / 255.0
y_test  = np.array(ds["test"]["label"])

### kPCA-RFF autoencoder

In [ ]:
Sigma = [5, 10, 20, 40]
num_rffs = 2000
num_comps = [100]
num_repeats = 10

methods = ['ours', 'bws', 'tr']
all_metrics, all_timing = [], []

for i in range(num_repeats):
    print(f"Repeat: {i+1}")
    print('-'*50)
    for sigma in Sigma:
        print(' '* 5 + f"sigma: {sigma}")
        gamma = 1 / (2 * sigma**2)

        for q in num_comps:
            print(' '* 10 + f"num comp: {q}")
            # fit
            kpca_rff = RFF_KernelPCA(n_components=q, n_rffs=num_rffs, gamma=gamma, random_state=42)
            _ = kpca_rff.fit(X_train)
            # transform
            X_transf = kpca_rff.transform(X_test)

            # reconstruction
            out_timing = {}
            out_metrics = {}

            for j, method in enumerate(methods):
                print(' '* 15 + f'method: {method}', end='\n' if j==len(methods)-1 else ' | ')
                alpha = 1e-6 if method == 'bws' else 0

                t = misc.TimingResult()
                with misc.timer_and_memory(t):
                    X_recon = getattr(kpca_rff, "inverse_transform_" + method)(X_transf, alpha=alpha)
                out_timing[method] = {'time_s': t.elapsed, 'peak_memory_mb': t.peak_kb/1024}
                out_metrics[method] = metrics(X_test.reshape(-1, 32,32), X_recon.reshape(-1, 32, 32))
            
            out_timing = out_timing | {'sigma': sigma, 'n_components':q, 'iter':i+1}
            out_metrics = out_metrics | {'sigma': sigma, 'n_components':q, 'iter':i+1}
            all_metrics.append(pd.DataFrame(out_metrics))
            all_timing.append(pd.DataFrame(out_timing))

    

In [ ]:
#==== time & memory
all_timing_df = pd.concat([pd.DataFrame(all_timing[i]) for i in range(len(all_timing))], axis=0).reset_index().rename(columns={'index': 'measure'})
# long format
all_timing_df = all_timing_df.melt(id_vars=["sigma", "n_components", "measure"], value_vars=methods, var_name="method")
# summarize across runs
all_timing_df = all_timing_df.groupby(["sigma", "n_components", "method", "measure"])["value"].agg(mean="mean", std="std").reset_index()

#======= performance
all_perf_df = pd.concat(all_metrics, axis=0)
all_perf_df['measure'] = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics)
# long format
all_perf_df = all_perf_df.melt(id_vars=["sigma", "n_components", "measure"], value_vars=methods, var_name="method")
# summarize across runs
all_perf_df = all_perf_df.groupby(["sigma", "n_components", "method", "measure"])["value"].agg(mean="mean", std="std").reset_index()

#=== merge
all_metrics_df = pd.concat([all_perf_df, all_timing_df], axis=0)

In [ ]:
df = all_metrics_df.pivot(index=['sigma', 'n_components', 'measure'], columns='method', values='mean').reset_index()
# only show a subset for space reasons
df = df[df["n_components"] == 100].drop(columns='n_components')

In [ ]:
def df_to_latex(df, methods=("bws", "ours", "tr"),
                caption=r"\textbf{CIFAR-10:} Reconstruction quality, runtime and memory.",
                label="tab:cifar10_results"):
    # one row per (sigma, measure); average if there are several seeds
    d = df.groupby(["sigma", "measure"], as_index=False)[list(methods)].mean()

    method_names = {"bws": r"B.W.S.~\cite{bakir2004learning}", "ours": "Ours", "tr": r"T.R.~\cite{tiddeman2023pre}"}
    measure_order = ["MSE", "SSIM", "PSNR", "time_s", "peak_memory_mb"]
    measure_names = {"MSE": "MSE", "SSIM": "SSIM", "PSNR": "PSNR (dB)",
                     "time_s": "Time (s)", "peak_memory_mb": "Memory (MB)"}
    lower_is_better = {"MSE": True, "SSIM": False, "PSNR": False,
                       "time_s": True, "peak_memory_mb": True}

    def fmt(measure, v, bold=False):
        if measure == "MSE":                              # a x 10^b
            mant, exp = f"{v:.1e}".split("e")
            s = f"{mant}\\times 10^{{{int(exp)}}}"
            return f"$\\mathbf{{{s}}}$" if bold else f"${s}$"
        s = f"{v:.2f}" if measure == "SSIM" else f"{v:.1f}"
        return f"\\textbf{{{s}}}" if bold else s

    lines = [
        "\\begin{table}[ht]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\begin{tabular}{cl" + "c" * len(methods) + "}",
        "\\toprule",
        "$\\sigma_{\\text{RBF}}$ & Measure & " + " & ".join(method_names[m] for m in methods) + " \\\\",
        "\\midrule",
    ]

    sigmas = sorted(d["sigma"].unique())
    for i, sigma in enumerate(sigmas):
        g = d[d["sigma"] == sigma].set_index("measure")
        first = True
        for meas in measure_order:
            if meas not in g.index:
                continue
            vals = g.loc[meas, list(methods)].astype(float).values
            best = vals.min() if lower_is_better[meas] else vals.max()
            best_str = fmt(meas, best)
            # bold every method that ties with the best at the displayed precision
            cells = [fmt(meas, v, bold=(fmt(meas, v) == best_str)) for v in vals]
            sigma_cell = f"{sigma:g}" if first else ""
            first = False
            lines.append(f"{sigma_cell} & {measure_names[meas]} & " + " & ".join(cells) + " \\\\")
        if i < len(sigmas) - 1:
            lines.append("\\midrule")

    lines += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    return "\n".join(lines)

# Save the generated table.
output_path = Path(out_dir) / "cifar10_table_reconstruction.tex"
output_path.write_text(df_to_latex(df), encoding="utf-8")

### Plot representative reconstruction for test data 

In [ ]:
# fit
q = [50, 100, 200]
data = {}
metrics_ = {}
sigma = 40
gamma = 1/(2*sigma**2)

for i in range(len(q)):
    print(f"kCA-RFF with {q[i]} components")
    # fit
    kpca_rff = RFF_KernelPCA(n_components=q[i], n_rffs=num_rffs, gamma=gamma, random_state=42)
    _ = kpca_rff.fit(X_train)
    # test
    data[str(q[i])] = kpca_rff.inverse_transform_ours(kpca_rff.transform(X_test), alpha=0.0)
    metrics_[str(q[i])] = metrics(X_test.reshape(-1, 32, 32), data[str(q[i])].reshape(-1, 32, 32))
    

In [23]:
GRID_N    = 5
IMG_SHAPE = (32, 32, 3)   # RGB
comp_keys = ["50", "100", "200"]

def fmt_metrics(m):
    return f"MSE: {m[0]:.1e} | SSIM: {m[1]:.2f} \nPSNR: {m[2]:.1f} dB"

def plot_image_grid(gs, images, nrows, ncols, title, panel_label, metrics_str=None):
    inner = gridspec.GridSpecFromSubplotSpec(
        nrows, ncols, subplot_spec=gs, wspace=0.04, hspace=0.04
    )
    for j, img in enumerate(images[:nrows * ncols]):
        ax = fig.add_subplot(inner[j // ncols, j % ncols])
        ax.imshow(
            np.clip(img.reshape(IMG_SHAPE), 0, 1), 
            aspect="equal",
        )
        ax.axis("off")

    anchor = fig.add_subplot(gs)
    anchor.set_axis_off()
    anchor.set_title(f"{panel_label} {title}", loc="left",
                     fontsize=8, fontweight="normal", pad=4)
    if metrics_str:
        anchor.text(0.5, -0.02, metrics_str, transform=anchor.transAxes,
                    fontsize=8, va="top", ha="center", style="italic")

fig = plt.figure(figsize=(5.5, 1.5))   # slightly taller for 32x32 images

outer_gs = gridspec.GridSpec(
    1, 4,
    figure=fig,
    wspace=0.10,
    left=0.02, right=0.98,
    top=0.88,  bottom=0.10,
)

plot_image_grid(
    outer_gs[0], X_test, GRID_N, GRID_N,
    title=r"Original", panel_label='', #r"(a)",
)

for i, q in enumerate(comp_keys):
    plot_image_grid(
        outer_gs[i + 1], data[q], GRID_N, GRID_N,
        title=rf"KPCA-RFF ($q={q}$)",
        panel_label='', #rf"({chr(98 + i)})",
        metrics_str=fmt_metrics(metrics_[q]),
    )

fig.savefig(out_dir + "figures/cifar10_reconstruction.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)